# JerryscanAI portable PatchCore training

This is the platform-independent GPU runner for the first controlled PatchCore comparison. It works on a college server, Lightning.ai, or another CUDA machine as long as that platform already has a compatible CUDA-enabled PyTorch and torchvision installation.

The notebook calls the canonical `training.models.train_patchcore` module. It verifies every dataset against `split_v2.csv`, fits with train and validation only, records the complete environment, and never passes the locked test directory to Anomalib. Hardware changes runtime and the safe batch size; it must not change the dataset split or experiment settings.

In [ ]:
from pathlib import Path

# If automatic discovery fails, set this to the cloned JerryscanAI directory.
PROJECT_ROOT_OVERRIDE = None

def find_project_root(start: Path) -> Path:
    for candidate in (start, *start.parents):
        if (candidate / "pyproject.toml").is_file() and (candidate / "training").is_dir():
            return candidate
    raise FileNotFoundError("Could not find JerryscanAI. Set PROJECT_ROOT_OVERRIDE.")

PROJECT_ROOT = (
    Path(PROJECT_ROOT_OVERRIDE).expanduser().resolve()
    if PROJECT_ROOT_OVERRIDE
    else find_project_root(Path.cwd().resolve())
)

# These are the only platform-specific paths you should normally edit.
DATA_ROOT = Path("/path/to/jerryscan-data/G01")
OUTPUT_ROOT = Path("/path/to/persistent-output/jerryscanai")
MANIFEST = PROJECT_ROOT / "data_manifests" / "G01" / "split_v2.csv"

VARIANTS = {
    "raw_letterbox_v1": DATA_ROOT / "raw_letterbox_v1",
    "fixed_crop_v1": DATA_ROOT / "fixed_crop_v1",
    "rembg_u2net_gray_v1": DATA_ROOT / "rembg_u2net_gray_v1",
    "rembg_u2net_black_v1": DATA_ROOT / "rembg_u2net_black_v1",
}
VARIANTS_TO_TRAIN = list(VARIANTS)

# Frozen stage-1 comparison settings. Keep them identical for all variants.
IMAGE_SIZE = 256
BATCH_SIZE = 8        # portable default; increase only for speed, never for one variant alone
EVAL_BATCH_SIZE = 1   # bounds PatchCore distance-matrix VRAM with the 10% memory bank
NUM_WORKERS = 4
CORESET_RATIO = 0.10
NUM_NEIGHBORS = 9
EMBEDDING_STORAGE = "cpu"  # keeps the full pre-coreset pool in system RAM
SEED = 42

assert MANIFEST.is_file(), MANIFEST
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
print("Project:", PROJECT_ROOT)
print("Data:", DATA_ROOT)
print("Outputs:", OUTPUT_ROOT)

In [ ]:
import importlib.metadata
import json
import subprocess
import sys

import torch
import torchvision

# PyTorch/torchvision are supplied by the GPU platform because their correct
# builds depend on its driver and CUDA generation. Do not replace a working pair.
torch_before = torch.__version__
torchvision_before = torchvision.__version__
print("Python:", sys.version)
print("PyTorch:", torch_before)
print("torchvision:", torchvision_before)
print("PyTorch CUDA build:", torch.version.cuda)
print("CUDA available:", torch.cuda.is_available())
assert torch.cuda.is_available(), (
    "Stop: select a GPU machine and install its compatible CUDA-enabled "
    "PyTorch/torchvision pair before continuing."
)

properties = torch.cuda.get_device_properties(0)
print("GPU:", properties.name)
print("VRAM GiB:", round(properties.total_memory / 1024**3, 2))

# Pin the model-training stack while preserving the platform's working torch.
# Anomalib depends on the GUI OpenCV distribution. On a headless Linux server
# that binary can fail while importing cv2 because libxcb is unavailable.
subprocess.run([
    sys.executable, "-m", "pip", "install",
    "--upgrade-strategy", "only-if-needed",
    "anomalib==2.2.0",
    "lightning==2.6.1",
    "openvino==2025.4.1",
    "opencv-python-headless==4.13.0.92",
    "pandas==2.3.3",
    "pillow==12.1.1",
], check=True)

# The GUI and headless OpenCV wheels install the same cv2 files. Remove GUI
# variants, then reinstall the pinned headless wheel so it owns those files.
subprocess.run([
    sys.executable, "-m", "pip", "uninstall", "-y",
    "opencv-python", "opencv-contrib-python",
], check=True)
subprocess.run([
    sys.executable, "-m", "pip", "install",
    "--force-reinstall", "--no-deps",
    "opencv-python-headless==4.13.0.92",
], check=True)

import cv2
import pandas as pd

print("OpenCV:", cv2.__version__)
print("pandas:", pd.__version__)
opencv_gui = next(
    line.split(":", 1)[1].strip()
    for line in cv2.getBuildInformation().splitlines()
    if line.strip().startswith("GUI:")
)
assert opencv_gui == "NONE", (
    "The active OpenCV build is not headless. Restart the kernel and rerun "
    "this cell before training."
)
assert pd.__version__ == "2.3.3", (
    "Anomalib 2.2 FolderDataset is incompatible with pandas 3.x. Restart "
    "the kernel and rerun this cell before training."
)

assert torch.__version__ == torch_before
assert torchvision.__version__ == torchvision_before

environment_file = OUTPUT_ROOT / "training_environment.freeze.txt"
freeze = subprocess.run(
    [sys.executable, "-m", "pip", "freeze"],
    check=True,
    text=True,
    capture_output=True,
).stdout
environment_file.write_text(freeze, encoding="utf-8")
print("Saved environment:", environment_file)

In [ ]:
def run(command, *, capture=False):
    print("$", " ".join(map(str, command)))
    return subprocess.run(
        [str(item) for item in command],
        cwd=PROJECT_ROOT,
        check=True,
        text=True,
        capture_output=capture,
    )

# Validate every folder and frozen sample identity before spending GPU time.
for preprocessing_id in VARIANTS_TO_TRAIN:
    dataset_root = VARIANTS[preprocessing_id]
    assert dataset_root.is_dir(), dataset_root
    run([
        sys.executable, "-m", "training.models.train_patchcore",
        "--dataset-root", dataset_root,
        "--manifest", MANIFEST,
        "--preprocessing-id", preprocessing_id,
        "--model-set", f"Patchcore_{preprocessing_id}_{IMAGE_SIZE}_c10_seed{SEED}",
        "--models-dir", OUTPUT_ROOT / "models",
        "--results-dir", OUTPUT_ROOT / "results",
        "--dry-run",
    ])
print("All selected datasets passed dry-run validation.")

In [ ]:
# Train the controlled stage-1 baselines sequentially.
# Batch size affects throughput, not the intended experiment. If CUDA OOM
# occurs, reduce BATCH_SIZE for every variant and restart the comparison.
for preprocessing_id in VARIANTS_TO_TRAIN:
    model_set = f"Patchcore_{preprocessing_id}_{IMAGE_SIZE}_c10_seed{SEED}"
    run([
        sys.executable, "-m", "training.models.train_patchcore",
        "--dataset-root", VARIANTS[preprocessing_id],
        "--manifest", MANIFEST,
        "--angle", "G01",
        "--preprocessing-id", preprocessing_id,
        "--model-set", model_set,
        "--image-size", IMAGE_SIZE,
        "--batch-size", BATCH_SIZE,
        "--eval-batch-size", EVAL_BATCH_SIZE,
        "--num-workers", NUM_WORKERS,
        "--accelerator", "gpu",
        "--coreset-sampling-ratio", CORESET_RATIO,
        "--num-neighbors", NUM_NEIGHBORS,
        "--embedding-storage", EMBEDDING_STORAGE,
        "--seed", SEED,
        "--models-dir", OUTPUT_ROOT / "models",
        "--results-dir", OUTPUT_ROOT / "results",
    ])

In [ ]:
# Confirm that every requested checkpoint has matching reproducibility metadata.
for preprocessing_id in VARIANTS_TO_TRAIN:
    model_set = f"Patchcore_{preprocessing_id}_{IMAGE_SIZE}_c10_seed{SEED}"
    folder = OUTPUT_ROOT / "models" / model_set
    checkpoint = folder / "G01.ckpt"
    metadata = folder / "G01.metadata.json"
    assert checkpoint.is_file(), checkpoint
    assert metadata.is_file(), metadata
    details = json.loads(metadata.read_text(encoding="utf-8"))
    assert details["dataset"]["test_used_during_training"] is False
    print(model_set, f"{checkpoint.stat().st_size / 1024**2:.1f} MiB", "OK")

print("Stage-1 training complete. Keep OUTPUT_ROOT on persistent storage.")